# Lecture 07 Practice Solutions - Logistic Regression

This notebook contains readable solutions for the separate practice notebook.


## Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def binary_cross_entropy(y, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


def softmax(z, axis=-1):
    shifted = z - np.max(z, axis=axis, keepdims=True)
    exp_z = np.exp(shifted)
    return exp_z / exp_z.sum(axis=axis, keepdims=True)


def multiclass_cross_entropy(target, probabilities):
    probabilities = np.clip(probabilities, 1e-12, 1 - 1e-12)
    return -np.sum(target * np.log(probabilities))


def train_logistic_regression(X, y, learning_rate=.1, epochs=500, lambda_=0.0):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    b = 0.0
    history = []
    for _ in range(epochs):
        p = sigmoid(X @ w + b)
        history.append(binary_cross_entropy(y, p) + lambda_ * np.sum(w ** 2))
        error = p - y
        dw = X.T @ error / n_samples + 2 * lambda_ * w
        db = np.mean(error)
        w -= learning_rate * dw
        b -= learning_rate * db
    return w, b, np.array(history)


rng = np.random.default_rng(7)
X0 = rng.normal([-1.1, -.8], [.75, .8], size=(70, 2))
X1 = rng.normal([1.0, .9], [.75, .8], size=(70, 2))
X_binary = np.vstack([X0, X1])
y_binary = np.r_[np.zeros(70), np.ones(70)]


## Exercise 1 - Sigmoid

Implement sigmoid and evaluate negative, zero, and positive scores.


In [ ]:
def sigmoid_exercise(z):
    return 1 / (1 + np.exp(-z))


print(sigmoid_exercise(np.array([-5.0, 0.0, 5.0])))


## Exercise 2 - Probabilities and Hard Classes

Convert scores into probabilities and use p >= 0.5 for hard classes.


In [ ]:
scores_ex2 = np.array([-3.0, -0.1, 0.1, 3.0])
probabilities_ex2 = sigmoid(scores_ex2)
classes_ex2 = (probabilities_ex2 >= .5).astype(int)
print("probabilities:", np.round(probabilities_ex2, 3))
print("classes:", classes_ex2)


## Exercise 3 - Binary Cross-Entropy

Implement the mean binary cross-entropy with numerical clipping.


In [ ]:
def bce_exercise(y, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


print(bce_exercise(np.array([1, 0, 1]), np.array([.9, .2, .4])))


## Exercise 4 - One Weight Update

Complete the gradient, parameter change, and new parameter values. The new loss
should be lower for this example.


In [ ]:
x_ex4 = np.array([2.0, 1.0])
y_ex4, lr_ex4 = 1.0, .1
w_ex4, b_ex4 = np.array([.1, -.2]), 0.0
p_ex4 = sigmoid(w_ex4 @ x_ex4 + b_ex4)
old_loss_ex4 = binary_cross_entropy(np.array([y_ex4]), np.array([p_ex4]))
error_ex4 = p_ex4 - y_ex4
dw_ex4 = error_ex4 * x_ex4
db_ex4 = error_ex4
delta_w_ex4 = -lr_ex4 * dw_ex4
delta_b_ex4 = -lr_ex4 * db_ex4
w_ex4_new = w_ex4 + delta_w_ex4
b_ex4_new = b_ex4 + delta_b_ex4
new_p_ex4 = sigmoid(w_ex4_new @ x_ex4 + b_ex4_new)
new_loss_ex4 = binary_cross_entropy(np.array([y_ex4]), np.array([new_p_ex4]))
print("old weights:", w_ex4, "gradient:", dw_ex4, "change:", delta_w_ex4)
print("new weights:", np.round(w_ex4_new, 3), "new bias:", round(b_ex4_new, 3))
print("old loss:", round(old_loss_ex4, 3))
print("new loss:", round(new_loss_ex4, 3))


## Exercise 5 - Batch Gradients

Complete the vectorized gradient expressions.


In [ ]:
X_ex5 = np.array([[1., 2.], [0., 1.], [2., -1.]])
y_ex5 = np.array([1., 0., 1.])
p_ex5 = sigmoid(X_ex5 @ np.array([.2, -.3]) + .1)
error_ex5 = p_ex5 - y_ex5
dw_ex5 = X_ex5.T @ error_ex5 / len(X_ex5)
db_ex5 = np.mean(error_ex5)
print("dw:", np.round(dw_ex5, 3), "db:", round(db_ex5, 3))


## Exercise 6 - Train the Model

Choose a learning rate and epoch count, run training, and inspect final
objective and training accuracy.


In [ ]:
w_ex6, b_ex6, history_ex6 = train_logistic_regression(
    X_binary, y_binary, learning_rate=.08, epochs=800
)
training_accuracy_ex6 = np.mean((sigmoid(X_binary @ w_ex6 + b_ex6) >= .5) == y_binary)
print(f"final objective: {history_ex6[-1]:.4f}")
print(f"training accuracy: {training_accuracy_ex6:.1%}")


## Exercise 7 - Stable Softmax

Complete softmax after shifting the logits.


In [ ]:
def softmax_exercise(z):
    shifted = z - np.max(z)
    exp_z = np.exp(shifted)
    return exp_z / exp_z.sum()


print(np.round(softmax_exercise(np.array([2.0, 1.0, 0.0])), 4))


## Exercise 8 - Multi-Class Cross-Entropy

Calculate the loss when horse is the true class.


In [ ]:
target_horse = np.array([0, 0, 1])
prediction_horse = np.array([.2, .3, .5])
loss_ex8 = multiclass_cross_entropy(target_horse, prediction_horse)
print(round(loss_ex8, 3))
